# Fase 1 — Da conversa importada à primeira chain

**Problema desta fase:** transformar o backup de WhatsApp (`_chat.txt`) em mensagens rastreáveis e, a partir delas, gerar **propostas estruturadas** (projeto, cliente, valor, compromisso) sem gravar nada como fato.

**Conceitos da disciplina demonstrados:** `ChatPromptTemplate`, modelo, `PydanticOutputParser`, composição LCEL (`prompt | modelo | parser`), `@chain`, `invoke`/`batch`/`stream`, saída estruturada validada.

**Decisões principais**
- Importação e parsing são **determinísticos** (Python + SQLite): mensagens multilinha, reimportação sem duplicar, ZIP lido em memória.
- O LLM só **interpreta**; cada proposta precisa citar IDs e trechos de mensagens-fonte, e o serviço revalida essas fontes.
- Um lote com saída inválida vira **falha explícita**; não há reparo infinito nem resposta inventada.

**Evidências reais (fora deste notebook):** [importação](../evidencias/marco1_importacao.json), [metadados da importação real](../evidencias/importacao_real_metadados.json), execuções reais da chain com a OpenAI ([1](../evidencias/marco2_api_real_20261110T020146555304Z.json), [2](../evidencias/marco2_api_real_20261110T021210978750Z.json), [3](../evidencias/marco2_api_real_20261110T021400551051Z.json)) e [fontes por proposta](../evidencias/marco2_fontes_por_proposta.json).

> **Fixtures:** este notebook roda **sem API**. As respostas do "modelo" são fixtures manuais identificadas e servem para testar o fluxo, não para provar compreensão do LLM. Bancos são temporários e só há dados fictícios.

Notebooks de origem (histórico do desenvolvimento): `incrementos/01_importacao_sqlite.ipynb`, `incrementos/02_extracao_estruturada.ipynb`.

---
*Origem desta parte: `incrementos/01_importacao_sqlite.ipynb`.*

## Marco 1 — Importação local e SQLite

Secretária IA, projeto acadêmico. Execute todas as células com o ambiente Python 3.12 preparado conforme README. Não usa OpenAI, LangSmith nem qualquer API. O núcleo é parsing determinístico e SQLite, ainda não uma chain/agente.

Somente dados fictícios são lidos. O banco desta demonstração é temporário e independente; executar novamente reconstrói o exemplo sem apagar um banco do usuário. O terminal usa um arquivo persistente em dados/privados.


In [1]:
from pathlib import Path
import json, sys, tempfile

# CAMINHOS PORTÁTEIS: localiza o projeto no diretório atual ou em seus pais, sem caminho pessoal.
raiz = None
for base in (Path.cwd(), *Path.cwd().parents):
    for candidato in (base, base / "Agentes Cognitivos com LangChain" / "Projeto final 2"):
        if (candidato / "PLANO_PROJETO_SECRETARIA_IA.md").is_file() and (candidato / "src/secretaria").is_dir():
            raiz = candidato
            break
    if raiz:
        break
if raiz is None:
    raise RuntimeError("Abra o notebook com a pasta completa do projeto e seu ambiente Python.")
sys.path.insert(0, str(raiz / "src"))
from secretaria.importador import ler_arquivo, interpretar
from secretaria.banco import Banco, Contexto
print("Python:", sys.version.split()[0])


Python: 3.12.6


### 1. Ler mensagens e preservar contexto

O cabeçalho inicia uma mensagem; linhas seguintes continuam o texto. Avisos, mídia omitida e mensagens removidas são identificados sem inventar conteúdo.


In [2]:
# PARSING: o gabarito é independente da interpretação, criado junto do corpus fictício.
arquivo = raiz / "dados/ficticios/conversa_empresa.txt"
gabarito = json.loads((raiz / "dados/ficticios/gabarito_importacao.json").read_text(encoding="utf-8"))
fonte = ler_arquivo(arquivo)
leitura = interpretar(fonte.texto)
assert len(leitura.mensagens) == gabarito["cabecalhos"]
assert leitura.mensagens[2].texto == gabarito["mensagem_multilinha"]["texto"]
print("Mensagens lidas:", len(leitura.mensagens), "| Rejeições:", len(leitura.rejeicoes))
print(leitura.mensagens[2].texto)


Mensagens lidas: 40 | Rejeições: 0
Projeto Alfa: implantação do portal de atendimento.
Cliente: Alfa Serviços.
Responsável: Ana.


> **Como ler o resultado:**
>
> - `Mensagens lidas: 40 | Rejeições: 0`: as 40 mensagens da conversa fictícia foram lidas, sem perda.
> - O texto impresso é **uma única mensagem de três linhas**: continua inteira, em vez de virar três mensagens.
> - Etapa sem IA: leitura determinística (parsing) em Python.

### 2. Criar banco e importar com autoria

A seleção de Ana é identidade de demonstração, não login real. O corpus bruto é privado do importador, com acesso do administrador. A autorização completa sobre dados operacionais pertence às próximas etapas.


In [3]:
# SQLITE/MIGRAÇÃO: o arquivo e o esquema nascem automaticamente, com FKs e transações.
ambiente_demo = tempfile.TemporaryDirectory(prefix="secretaria_notebook_")
caminho_banco = Path(ambiente_demo.name) / "demo.sqlite3"
ctx = Contexto("demo", "ana")
with Banco(caminho_banco) as banco:
    banco.preparar_demo()
    corpus_id = banco.selecionar_corpus(ctx, "Empresa fictícia")
    primeira = banco.importar(ctx, corpus_id, arquivo)
    assert primeira["novas"] == 40
    print(primeira)


{'importacao_id': 1, 'corpus_id': 1, 'reimportacao': False, 'lidas': 40, 'novas': 40, 'reutilizadas': 0, 'rejeicoes': 0, 'avisos': 1, 'tipos': {'sistema': 1, 'mensagem': 37, 'midia_omitida': 1, 'removida': 1}, 'formatos': ['ios'], 'fuso': 'America/Sao_Paulo', 'origem_fuso': 'premissa configurada; não informada no arquivo', 'primeiro_instante_utc': '2026-11-09T11:00:00+00:00', 'ultimo_instante_utc': '2026-11-09T11:39:00+00:00'}


> **Como ler o resultado:**
>
> - `'novas': 40`: as mensagens foram gravadas no SQLite.
> - `'tipos'`: separa mensagem comum (37), aviso do sistema (1), mídia omitida (1) e mensagem apagada (1).

### 3. Reimportar sem duplicar

O hash do conteúdo impede repetir o lote. A identidade das mensagens combina corpus, data, autor, conteúdo e ocorrência, preservando duas mensagens legítimas idênticas.


In [4]:
# PERSISTÊNCIA: reabre o mesmo arquivo após fechar a conexão anterior.
with Banco(caminho_banco) as banco:
    segunda = banco.importar(ctx, corpus_id, arquivo)
    assert segunda["reimportacao"] and segunda["novas"] == 0
    assert banco.contar(ctx, corpus_id) == 40
    repetidas = [m for m in banco.listar(ctx, corpus_id) if m["texto"] == gabarito["repeticao_legitima"]["texto"]]
    assert len(repetidas) == 2
    print("Novas na segunda importação:", segunda["novas"])
    print("Mensagens preservadas:", banco.contar(ctx, corpus_id))
    print("Ocorrências legítimas idênticas:", len(repetidas))


Novas na segunda importação: 0
Mensagens preservadas: 40
Ocorrências legítimas idênticas: 2


> **Como ler o resultado:**
>
> - `Novas na segunda importação: 0`: importar o mesmo arquivo de novo **não duplica**.
> - `Ocorrências legítimas idênticas: 2`: a mesma frase enviada duas vezes de verdade é preservada.

### 4. Mesmo TXT em ZIP e rastreabilidade

O ZIP é lido em memória e anexos não são extraídos. Os avisos de edição não alteram o original nem escolhem automaticamente qual interpretação vale.


In [5]:
from zipfile import ZipFile
# FONTES: ZIP e TXT com o mesmo conteúdo apontam ao mesmo lote do corpus.
zip_demo = Path(ambiente_demo.name) / "exportacao.zip"
with ZipFile(zip_demo, "w") as z:
    z.write(arquivo, arcname="_chat.txt")
with Banco(caminho_banco) as banco:
    terceira = banco.importar(ctx, corpus_id, zip_demo)
    assert terceira["reimportacao"] and banco.contar(ctx, corpus_id) == 40
    avisos = banco.avisos(ctx, corpus_id, primeira["importacao_id"])
    assert any(a["tipo"] == "possivel_edicao" for a in avisos)
    print("ZIP repetido:", terceira["novas"], "novas mensagens")
    print("Avisos:", avisos)


ZIP repetido: 0 novas mensagens
Avisos: [{'tipo': 'possivel_edicao', 'detalhes': {'mensagem_nova': 30, 'candidatas_anteriores': [29], 'motivo': 'Edição sinalizada; original preservado e resolução manual pendente.'}}]


> **Como ler o resultado:**
>
> - `ZIP repetido: 0 novas mensagens`: o mesmo conteúdo em ZIP é reconhecido como repetido.
> - `possivel_edicao`: a mensagem 30 foi editada e parece corrigir a 29. O sistema **avisa**, mas não escolhe sozinho qual vale.

### O que foi demonstrado e o que falta

Demonstrado: parsing, fonte/linhas, criação do SQLite, autoria de importação, persistência entre conexões, repetição sem duplicar e leitura segura de ZIP. Não foi demonstrada memória cognitiva, compreensão por LLM, agenda, financeiro operacional, RAG, API externa nem LangSmith. A persistência aqui não comprova a conservação de um runtime temporário após reiniciar o notebook.

Próximo incremento: propostas estruturadas a partir do corpus fictício, sem gravação automática de fatos operacionais.


---
*Origem desta parte: `incrementos/02_extracao_estruturada.ipynb`.*

## Marco 2 — Primeira chain de propostas com fontes

Execute com requirements-notebook.txt e requirements-ia.txt instalados no mesmo kernel.
O notebook é offline por padrão: usa respostas manuais identificadas, não comprova compreensão de LLM.
A opção OpenAI no terminal usa a mesma chain com gpt-4o-mini, sem enviar o backup real.
Propostas ficam aguardando revisão; nenhuma agenda, pagamento ou projeto operacional é gravado.


In [6]:
from pathlib import Path
import json, sys, tempfile

# CAMINHOS PORTÁTEIS: localiza o projeto no diretório atual ou em seus pais, sem caminho pessoal.
raiz = None
for base in (Path.cwd(), *Path.cwd().parents):
    for candidato in (base, base / "Agentes Cognitivos com LangChain" / "Projeto final 2"):
        if (candidato / "PLANO_PROJETO_SECRETARIA_IA.md").is_file() and (candidato / "src/secretaria").is_dir():
            raiz = candidato
            break
    if raiz:
        break
if raiz is None:
    raise RuntimeError("Abra o notebook com a pasta completa do projeto e seu ambiente Python.")
sys.path.insert(0, str(raiz / "src"))
from secretaria.importador import ler_arquivo, interpretar
from secretaria.banco import Banco, Contexto
print("Python:", sys.version.split()[0])

from secretaria.chain_extracao import criar_chain
from secretaria.extracao import preparar_lotes, analisar_snapshot
from secretaria.fixtures_extracao import criar_modelo_fixture
from secretaria.apresentacao import exibir_analise


Python: 3.12.6


### 1. Snapshot completo e lotes com vizinhos

SQLite fornece todos os registros autorizados. Lotes de oito mensagens têm uma mensagem vizinha de cada lado.
É cobertura do processamento, não afirmação de que o modelo entendeu todas as mensagens.


In [7]:
# SQLITE/CONTEXTO: corpus fictício e identidade fornecida pela aplicação.
ambiente_demo = tempfile.TemporaryDirectory(prefix="secretaria_chain_notebook_")
banco_arquivo = Path(ambiente_demo.name) / "demo.sqlite3"
ctx = Contexto("demo", "ana")
with Banco(banco_arquivo) as banco:
    banco.preparar_demo()
    corpus = banco.selecionar_corpus(ctx, "Demonstração fictícia")
    banco.importar(ctx, corpus, raiz / "dados/ficticios/conversa_empresa.txt")
    snapshot = banco.snapshot_analise(ctx, corpus)
lotes = preparar_lotes(snapshot)
assert len(snapshot["mensagens"]) == 40 and len(lotes) == 5
assert [m["id"] for l in lotes for m in l["mensagens_foco"]] == [m["id"] for m in snapshot["mensagens"]]
print("40 mensagens em 5 lotes, sem truncamento.")


40 mensagens em 5 lotes, sem truncamento.


> **Como ler o resultado:**
>
> - `40 mensagens em 5 lotes`: a conversa foi dividida em 5 lotes de 8 para a chain. Nenhuma mensagem ficou de fora.

### 2. ChatPromptTemplate, modelo e parser

LCEL liga prompt | modelo | PydanticOutputParser. O decorador @chain demonstra a mesma sequência imperativa.
O parser prova estrutura; os IDs/trechos/valores ainda passam pela validação de fontes no serviço.


In [8]:
# RUNNABLE/INVOKE: o modelo injetado aqui é fixture manual, não OpenAI.
entrada = {"pacote": json.dumps(lotes[0], ensure_ascii=False)}
lcel = criar_chain(criar_modelo_fixture(), "lcel")
imperativa = criar_chain(criar_modelo_fixture(), "imperativa")
a = lcel.invoke(entrada)
b = imperativa.invoke(entrada)
assert a.model_dump() == b.model_dump()
print("LCEL e @chain produziram o mesmo contrato na fixture.")


LCEL e @chain produziram o mesmo contrato na fixture.


> **Como ler o resultado:**
>
> - A chain é `prompt | modelo | parser` (montada em `src/secretaria/chain_extracao.py`).
> - As duas formas de montar (LCEL e `@chain`) deram o **mesmo resultado**. O modelo aqui é fixture (resposta pronta, sem API).

### 3. Batch e stream de objetos completos

batch executa as leituras independentes. Nesta implementação imperativa, stream emite um objeto completo;
não demonstra streaming de tokens nem exibe propostas parciais como fatos verificados.


In [9]:
# RUNNABLE/BATCH/STREAM: demonstra métodos reais, com respostas roteirizadas identificadas.
respostas = lcel.batch([{"pacote": json.dumps(l, ensure_ascii=False)} for l in lotes], config={"max_concurrency": 2})
partes = list(imperativa.stream(entrada))
assert len(respostas) == 5 and len(partes) == 1
print("batch:", len(respostas), "lotes; stream imperativo:", len(partes), "objeto completo.")


batch: 5 lotes; stream imperativo: 1 objeto completo.


> **Como ler o resultado:**
>
> - `batch`: processa os 5 lotes de uma vez. `stream`: entrega o resultado em partes (aqui, um objeto completo).

### 4. Propostas, fontes e cobertura

Condição de entrada é diferente de pagamento informado. Visita confirmada na conversa é diferente de registro
gravado na agenda. Algumas interpretações ficam pendentes; não escolhemos silenciosamente a última versão.


In [10]:
# VALIDAÇÃO: o serviço confere fontes, valores escritos e cobertura antes da apresentação.
resultado = analisar_snapshot(snapshot, criar_modelo_fixture(), "fixture_manual")
assert resultado["status"] == "completa"
assert resultado["mensagens_em_lotes_concluidos"] == 40
assert len(resultado["propostas"]) == 8
assert all(p["status"] == "aguardando_revisao" for p in resultado["propostas"])
assert resultado["gravacoes_operacionais"] == 0
exibir_analise(resultado)
with Banco(banco_arquivo) as banco:
    assert banco.contar(ctx, corpus) == 40
    assert banco.con.execute("SELECT COUNT(*) FROM importacoes").fetchone()[0] == 1



PROPOSTAS PARA REVISÃO — nenhum registro operacional foi gravado
Origem: fixture_manual | Composição: lcel
Cobertura dos lotes: 40 / 40
[FIXTURE LOCAL] Respostas manuais; nenhuma chamada de LLM.

1. PROJETO | Alfa | relato
Implantação do portal; responsável informado Ana.
Fonte M0003: Projeto Alfa: implantação do portal de atendimento.
Cliente: Alfa Serviços.
Responsável: Ana.

2. FINANCEIRO | Alfa | previsto
Orçamento Alfa informado em R$ 10.000,00.
Tipo: orcamento_proposto
Valor: R$ 10.000,00
Fonte M0004: Orçamento Alfa: R$ 10.000,00. Entrada combinada de 30%; restante em 20/11/2026.

3. FINANCEIRO | Alfa | previsto
Entrada combinada de 30%; não comprova pagamento.
Tipo: condicao_prevista
Percentual informado: 30%
Fonte M0004: Orçamento Alfa: R$ 10.000,00. Entrada combinada de 30%; restante em 20/11/2026.

4. AGENDA | Alfa | confirmacao
Visita Alfa confirmada na conversa; local ainda pendente.
Início: 2026-11-10T14:00:00-03:00 | Fim: 2026-11-10T15:00:00-03:00
Fonte M0008: Confirmo v

> **Como ler o resultado:**
>
> - São **propostas**, não registros: nada foi gravado na agenda ou no financeiro.
> - Cada proposta mostra: área | cliente | estado, e a **mensagem de origem** (`M0004` = mensagem 4).
> - **previsto** (orçamento) ≠ **relato** ("recebi R$ 3.000" não é pagamento confirmado).
> - `MENSAGEM EDITADA`: visita Gama às 15h ou 16h? O sistema não escolhe: pede revisão humana.
> - `Cobertura dos lotes: 40 / 40`: todas as mensagens passaram pela chain.

### 5. Contrato inválido vira falha explícita

A análise não passa a ser completa se um lote falhar. Não há reparo infinito nem resposta inventada como fallback.


In [11]:
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda
# PARSER/ERROS: caso adverso local, não uma falha observada de API real.
invalido = RunnableLambda(lambda _: AIMessage(content="não é JSON"))
pequeno = {**snapshot, "mensagens": snapshot["mensagens"][:1]}
falha = analisar_snapshot(pequeno, invalido, "fixture_adversa")
assert falha["status"] == "parcial" and falha["propostas"] == []
print("Falha identificada:", falha["lotes"][0]["tipo_erro"])


Falha identificada: OutputParserException


> **Como ler o resultado:**
>
> - Simulação de resposta fora do formato ("não é JSON"): o parser **recusa** (`OutputParserException`) e a análise fica **parcial**, sem inventar resposta.

### Limites e próximo passo

Isto é uma chain de interpretação, não um agente com ferramentas. Oito propostas são respostas manuais
do modo fixture, não gabarito de todas as informações possíveis. Fontes e valores consistentes não provam
toda a interpretação semântica; revisão humana e verificador completo vêm em incrementos posteriores.
Múltiplos lotes podem gerar propostas semanticamente repetidas. Não foi implementada consolidação final
nem aplicação de confirmação. O corpus real privado não é selecionado para envio externo neste fluxo.

Para chamada real: opção 5 no terminal ou --analisar api --arquivo-env caminho/.env.
Não coloque chave no notebook nem em arquivos versionados.
